# Training the synthetic-speech detector on ASVspoof 2019 LA

**Information Security Project - Sri Lanka Institute of Information Technology**

This notebook trains the detector on the **real** ASVspoof 2019 Logical Access
benchmark and produces the numbers that belong in the report.

There is deliberately **no synthetic-data fallback in this notebook.** An
earlier version of the pipeline silently generated 300 oscillator tones when it
could not find the dataset and trained on those instead. The resulting
checkpoint scored near-perfectly on a trivially separable toy problem, and
those figures were then carried into the report as detection accuracy. If the
dataset is missing here, the notebook stops and tells you.

Runtime: **Runtime -> Change runtime type -> T4 GPU** before you start.


## 1. Confirm the GPU

Training the full training partition (25,380 utterances) takes roughly 25-40
minutes per epoch on CPU and about 1-2 minutes on a T4. Do not attempt this
without a GPU runtime.

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU"


## 2. Install dependencies and fetch the project code

Upload the `voxshield` folder to your Google Drive, or clone it from your
group's repository. The notebook imports the project's own modules so that the
feature extraction used in training is byte-for-byte the feature extraction the
API uses at inference time. Re-implementing the preprocessor inside the
notebook is how training/serving skew gets introduced.

In [ ]:
!pip install -q librosa soundfile scikit-learn matplotlib tqdm

from google.colab import drive
drive.mount('/content/drive')

# Adjust to wherever you placed the project.
PROJECT_DIR = '/content/drive/MyDrive/voxshield'

import sys, os
assert os.path.isdir(PROJECT_DIR), f"{PROJECT_DIR} not found - upload the project folder first"
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import config
from backend.preprocessor import AudioPreprocessor
from model import datasets
from model.cnn_architecture import VoiceDeepfakeCNN, count_parameters
from model.evaluate import evaluate_scores, per_attack_eer, save_roc_curve, format_report
from model.train import train_model, score_corpus, write_metadata

print('project modules loaded from', PROJECT_DIR)


## 3. Obtain ASVspoof 2019 LA

The corpus is distributed by the University of Edinburgh's DataShare
repository. Open the landing page, accept the licence, and copy the current
download link for `LA.zip` - the direct URL changes between releases, so check
it rather than trusting a link copied from a tutorial:

> https://datashare.ed.ac.uk/handle/10283/3336

The Logical Access partition is roughly 7-8 GB compressed. Colab's disk is
ephemeral, so either download it into the session each time or unzip it once
into your Drive.

The expected layout after extraction is:

```
LA/
  ASVspoof2019_LA_train/flac/*.flac          25,380 utterances, attacks A01-A06
  ASVspoof2019_LA_dev/flac/*.flac            24,844 utterances, attacks A01-A06
  ASVspoof2019_LA_eval/flac/*.flac           71,237 utterances, attacks A07-A19
  ASVspoof2019_LA_cm_protocols/*.txt         labels
```

The split matters and is worth stating explicitly: **the evaluation
partition contains spoofing systems that never appear in training.** That is
the whole point of the benchmark - it measures generalisation to unseen attacks
rather than memorisation of familiar ones.

In [ ]:
# Option A - unzip a copy you already placed in Drive
# !unzip -q '/content/drive/MyDrive/datasets/LA.zip' -d /content/

# Option B - download into the session (paste the current link from DataShare)
# !wget -q --show-progress '<PASTE LA.zip LINK HERE>' -O /content/LA.zip
# !unzip -q /content/LA.zip -d /content/

DATA_ROOT = '/content/LA'

import os
if not os.path.isdir(DATA_ROOT):
    raise SystemExit(
        'ASVspoof 2019 LA is not present at ' + DATA_ROOT + '.\n'
        'This notebook will not substitute generated audio for the corpus. '
        'Download the dataset and re-run this cell.'
    )
print('dataset found')


## 4. Build the corpora

`limit` subsamples in a class-stratified way that keeps every attack type
present. Use it for a fast first run; remove it for the numbers you report.

ASVspoof LA is about 10% bonafide and 90% spoof. That imbalance is why the
training loop weights the loss and why the report should lead with EER rather
than accuracy.

In [ ]:
LIMIT = None      # e.g. 6000 for a quick run; None for the full partition

train_corpus = datasets.asvspoof_2019_la(DATA_ROOT, 'train', limit=LIMIT)
dev_corpus   = datasets.asvspoof_2019_la(DATA_ROOT, 'dev',   limit=LIMIT)
eval_corpus  = datasets.asvspoof_2019_la(DATA_ROOT, 'eval',  limit=LIMIT)

for corpus in (train_corpus, dev_corpus, eval_corpus):
    print(corpus.summary())


## 5. Inspect the features before training

Always look at a spectrogram from each class before committing GPU hours. If
the two classes look identical here, or if one of them is blank, the bug is in
the front-end and no amount of training will fix it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

pre = AudioPreprocessor()

bona = next(f for f, y in zip(train_corpus.files, train_corpus.labels) if y == 0)
spoof = next(f for f, y in zip(train_corpus.files, train_corpus.labels) if y == 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=110)
for ax, path, title in zip(axes, (bona, spoof), ('bonafide', 'spoof')):
    ax.imshow(pre.process_single(path)[0], aspect='auto', origin='lower', cmap='inferno')
    ax.set_title(title)
    ax.set_xlabel('time frame')
    ax.set_ylabel('mel bin')
plt.tight_layout()
plt.show()


## 6. Train

The training loop lives in `model/train.py`, not in this notebook, so the same
code runs on a laptop and on Colab and there is one place to fix a bug.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

model, result = train_model(
    train_corpus, dev_corpus,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_workers=2,
    out_dir=os.path.join(PROJECT_DIR, 'model', 'saved_models'),
)

best = result['best']
threshold = best['eer_threshold']
print('\nDEVELOPMENT PARTITION')
print(format_report(best))


## 7. Evaluate on the held-out partition

The threshold is carried over from the development partition rather than
re-optimised here. Choosing the threshold on the same data you report against
is a form of test-set leakage, and it is the single easiest way to publish a
number you cannot reproduce in deployment.

In [ ]:
device = next(model.parameters()).device
y_true, y_scores, attacks = score_corpus(model, eval_corpus, pre, device, batch_size=64)

eval_metrics = evaluate_scores(y_true, y_scores, threshold=threshold)
attack_breakdown = per_attack_eer(y_true, y_scores, attacks)

print('EVALUATION PARTITION (unseen attacks A07-A19)')
print(format_report(eval_metrics, attack_breakdown))


### Reading the per-attack table

Expect the numbers to vary widely between attack types. In published ASVspoof
work, A17 (a waveform-filtering voice conversion system) is the attack that
most countermeasures struggle with, while several of the neural TTS attacks are
close to solved. A row that is dramatically worse than the rest is a *result*,
not a failure - it identifies the residual risk in the deployed system, and
saying so is more convincing than a single flattering average.

In [ ]:
roc_path = os.path.join(PROJECT_DIR, 'docs', 'roc_curve_asvspoof.png')
os.makedirs(os.path.dirname(roc_path), exist_ok=True)
save_roc_curve(y_true, y_scores, roc_path,
               title='ASVspoof 2019 LA evaluation - ROC')

from IPython.display import Image, display
display(Image(roc_path))


## 8. Write the provenance record

This is the step that lets the API describe itself honestly. The metadata file
records the corpus, the calibrated threshold and the measured metrics. Without
it, the serving runtime refuses to call the checkpoint trained and shows a
warning banner in the dashboard.

In [ ]:
meta_path = os.path.join(PROJECT_DIR, 'model', 'saved_models', 'model_meta.json')

write_metadata(
    __import__('pathlib').Path(meta_path),
    train_corpus,
    best,
    threshold,
    eval_metrics,
    attack_breakdown,
    epochs=len(result['history']),
)

print(open(meta_path).read()[:1500])


## 9. Copy the artefacts back

Download these two files into `model/saved_models/` in the project you run
locally. Both are required - the weights alone will load as an untrusted
checkpoint.

* `voice_deepfake_cnn.pth`
* `model_meta.json`

Also keep `docs/roc_curve_asvspoof.png` for the report.

In [ ]:
from google.colab import files
files.download(os.path.join(PROJECT_DIR, 'model', 'saved_models', 'voice_deepfake_cnn.pth'))
files.download(meta_path)
files.download(roc_path)


## 10. What to record in the logbook

For each run, note: the partition sizes actually used, the number of epochs,
the development EER, the evaluation EER at the fixed threshold, and the worst
per-attack EER. A run that is not written down did not happen, and the logbook
is separately assessed.